In [1]:
from collections import Counter
import json
import os
import sys

path = "/home/lk3591/Documents/code/RawByteClf"
os.chdir(path)
if path not in sys.path:
	sys.path.insert(0, path)

from src.data.utils import stream_sorel_meta
from src.data.detect_packing_sorel import DIEC_MODES

Entered __file__='/home/lk3591/Documents/code/RawByteClf/src/__init__.py'


/home/lk3591/miniconda3/envs/RawByteClf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def nullable_nested_dictionary_to_non_nullable_nested_dictionary(d):
    for k in d:
        if d[k] is None:
            d[k] = {}
        elif isinstance(d[k], dict):
            nullable_nested_dictionary_to_non_nullable_nested_dictionary(d[k])

In [3]:
p = "/home/lk3591/Documents/datasets/Sorel/diec/consolidated/output.json"
d = json.load(open(p))
nullable_nested_dictionary_to_non_nullable_nested_dictionary(d)
total = len(d)
total

7900000

In [6]:
# No mode detects packing
v = sum(1 for r in d.values() if all(not m.get("packed", False) for m in r.values()))
v, round(v / total, 4)

(1758372, 0.2226)

In [7]:
# All modes detects packing
v = sum(1 for r in d.values() if all(m.get("packed", False) for m in r.values()))
v, round(v / total, 2)

(1363557, 0.17)

In [8]:
# Any mode detects packing
v = sum(1 for r in d.values() if any(m.get("packed", False) for m in r.values()))
v, round(v / total, 2)

(6141628, 0.78)

In [35]:
# Heuristic but not signature
v = sum(1 for r in d.values() if (not (r["recursive"].get("packed", False) or r["deep"].get("packed", False)) and (r["heuristic"].get("packed", False))))
v, round(v / total, 2)

(1143379, 0.6)

In [48]:
# Signature or signature and heuristic
v = sum(1 for r in d.values() if ((r["recursive"].get("packed", False) or r["deep"].get("packed", False))))
v, round(v / total, 4)

(330383, 0.1739)

In [37]:
# Signature but not heuristic
v = sum(1 for r in d.values() if ((r["recursive"].get("packed", False) or r["deep"].get("packed", False)) and (not r["heuristic"].get("packed", False))))
v, round(v / total, 4)

(232, 0.0001)

In [28]:
# Recursive/deep sigatures agree/disagree
def agree(r):
    if r["recursive"] == {} or r["deep"] == {}:
        return None
    return r["recursive"]["packed"] == r["deep"]["packed"]

sum(1 for r in d.values() if agree(r) is True), sum(1 for r in d.values() if agree(r) is False)

(1889555, 2090)

In [59]:
# Recursive/deep sigatures but not known
v = sum(
    1 for r in d.values()
    if (r["recursive"].get("packed", False) and r["recursive"].get("packer", "") == "")
    and (r["deep"].get("packed", False) and r["deep"].get("packer", "") == "")
)
v, round(v / total, 2)

(0, 0.0)

In [9]:
packers = []
for r in d.values():
    p = []
    for r in r.values():
        if r.get("packed", False):
            p.extend(r.get("packer", []))
    packers.extend(list(set(p)))

packers = Counter(packers)
packers

Counter({'Heursitic': 6138088,
         'UPX': 888835,
         'ASPack': 148525,
         'MPRESS': 126190,
         'Petite': 96563,
         'EP:MPRESS': 53321,
         'PECompact': 37491,
         'DxPack': 34187,
         'NeoLite': 22445,
         '(Win)Upack': 8324,
         'MEW': 5266,
         'NsPacK': 1748,
         'kkrunchy': 1531,
         'Packman': 804,
         'MoleBox': 603,
         'FSG': 561,
         'PyInstaller': 297,
         'BeRo': 274,
         'Spoon Studio': 267,
         '.NETZ': 260,
         'Exe32Pack': 143,
         'SCREEN2EXE/SCREEN2SWF': 66,
         'PE-PACK': 59,
         'RLPack': 59,
         'AHpacker': 38,
         'EXEPACK': 38,
         'KByS Packer': 32,
         'nPack': 27,
         'PKLITE32': 23,
         'Software Compress': 14,
         'ezip': 14,
         'JDPack': 12,
         'NakedPacker': 12,
         'PKLITE': 11,
         'Pack Master': 7,
         'WWPACK': 7,
         'XComp': 7,
         'mPack': 6,
         'VPacker': 

: 

In [61]:
packers = []
for r in d.values():
    p = []
    for k, r in r.items():
        if k not in ("recursive", "deep"):
            continue
        if r.get("packed", False):
            p.extend(r.get("packer", "").split("|"))
    packers.extend(list(set(p)))

packers = Counter(packers)
len(packers), packers

(40,
 Counter({'UPX': 213631,
          'ASPack': 35502,
          'MPRESS': 30247,
          'Petite': 23244,
          'EP:MPRESS': 12850,
          'PECompact': 9056,
          'DxPack': 8257,
          'NeoLite': 5479,
          '(Win)Upack': 1966,
          'MEW': 1336,
          'NsPacK': 445,
          'kkrunchy': 413,
          'Packman': 196,
          'MoleBox': 154,
          'FSG': 151,
          'BeRo': 68,
          '.NETZ': 61,
          'PyInstaller': 56,
          'Spoon Studio': 50,
          'Exe32Pack': 40,
          'SCREEN2EXE/SCREEN2SWF': 22,
          'PE-PACK': 21,
          'RLPack': 20,
          'AHpacker': 10,
          'nPack': 8,
          'EXEPACK': 8,
          'KByS Packer': 7,
          'PKLITE32': 6,
          'NakedPacker': 6,
          'WWPACK': 4,
          'ezip': 3,
          'Software Compress': 3,
          'Pack Master': 2,
          'XPACK': 2,
          'ANDpakk': 2,
          'PKLITE': 1,
          'mPack': 1,
          'VPacker': 1,
     

In [51]:
213633 / total

0.11246641927174501

In [54]:
v = sum(1 for s in stream_sorel_meta() if s.packed > 0)
v / total

1.961572012400982

In [56]:
v / 10e6

0.3726059